## __build_po_dataframe() 함수

In [153]:
import pandas as pd
import glob
import os
from sap_download import download_stockout_prediction, download_inventory_overview
from datetime import datetime

In [154]:
df_stockout = pd.read_excel("psi_input/품절예상조회/품절예상조회_04.06 14시30분.xlsx")

In [155]:
df_overview = pd.read_excel("psi_input/재고개요/재고개요_04.06 14시30분.xlsx")

In [156]:
df_sf = pd.read_excel("psi_input/SF/SF_2603.xlsx")
df_sf.columns = df_sf.columns.astype(str)

In [157]:
df_stockout = df_stockout[["자재", "자재명", "3개월 평균출하", "당월출하"]].copy()
df_stockout.rename(columns = {"자재":"자재코드", "자재명":"자재내역", "3개월 평균출하":"3평판"}, inplace=True)
df_stockout["자재코드"] = df_stockout["자재코드"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
df_stockout["당월출하"] = pd.to_numeric(df_stockout["당월출하"], errors="coerce").fillna(0)
df_stockout["3평판"] = pd.to_numeric(df_stockout["3평판"], errors="coerce").fillna(0)

In [158]:
today = datetime.today() 
year_month = today.strftime('%Y.%m')
df_sf = df_sf[["자재코드", "자재내역", "관리 채널", year_month]].copy()
df_sf.rename(columns = {"관리 채널":"관리채널", year_month:"SF"}, inplace=True)
df_sf["자재코드"] = df_sf["자재코드"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
df_sf["SF"] = pd.to_numeric(df_sf["SF"], errors="coerce").fillna(0)
df_sf = df_sf.groupby("자재코드").agg(자재내역=("자재내역", "first"), SF=("SF", "sum")).reset_index()
df_sf["SF"] = df_sf["SF"].round().astype(int)

In [159]:
df_standard = pd.merge(df_stockout, df_sf, on="자재코드", how="outer", suffixes=("", "_sf"))
df_standard["자재내역"] = df_standard["자재내역"].fillna(df_standard["자재내역_sf"])
df_standard.drop(columns=["자재내역_sf"], inplace=True)

df_standard["3평판"]   = df_standard["3평판"].fillna(0)
df_standard["당월출하"] = df_standard["당월출하"].fillna(0)
df_standard["SF"]     = df_standard["SF"].fillna(0)

In [160]:
display(df_standard.head())

,자재코드,자재내역,3평판,당월출하,SF
0,2303084,[EA]이지듀멜라비토닝미스트로즈쇼핑백,0.0,0.0,1023.0
1,2303248,이지듀멜라비토닝원데이앰플로즈에디션_세트단상자(박스형),0.0,0.0,0.0
2,2303830,[부자재]기미팩트세트(본품3+리필3+퍼프3)_싸바리(공용),0.0,0.0,0.0
3,7000301,케어트로핀 니들스왑셋트 1ea,0.0,0.0,0.0
4,7000302,케어트로핀 펜주사기셋트 1ea (내수),0.0,0.0,0.0


In [161]:
df_standard["판매율(평판)"] = (df_standard["당월출하"] / df_standard["3평판"]).where(df_standard["3평판"] != 0)
df_standard["판매율(SF)"]  = (df_standard["당월출하"] / df_standard["SF"]).where(df_standard["SF"] != 0)

df_standard["3평판"] = df_standard["3평판"].astype(int)
df_standard["당월출하"] = df_standard["당월출하"].astype(int)
df_standard["SF"] = df_standard["SF"].astype(int)

In [162]:
display(df_standard.head())

,자재코드,자재내역,3평판,당월출하,SF,판매율(평판),판매율(SF)
0,2303084,[EA]이지듀멜라비토닝미스트로즈쇼핑백,0,0,1023,NaN,0.0
1,2303248,이지듀멜라비토닝원데이앰플로즈에디션_세트단상자(박스형),0,0,0,NaN,NaN
2,2303830,[부자재]기미팩트세트(본품3+리필3+퍼프3)_싸바리(공용),0,0,0,NaN,NaN
3,7000301,케어트로핀 니들스왑셋트 1ea,0,0,0,NaN,NaN
4,7000302,케어트로핀 펜주사기셋트 1ea (내수),0,0,0,NaN,NaN


In [163]:
df_standard = df_standard.sort_values("판매율(SF)", ascending=False, na_position="last").reset_index(drop=True)
df_standard = df_standard[["자재코드", "자재내역", "3평판", "SF", "당월출하", "판매율(평판)", "판매율(SF)"]]

In [164]:
display(df_standard.head())

,자재코드,자재내역,3평판,SF,당월출하,판매율(평판),판매율(SF)
0,9302971,(단종)[임가공]RX_프레좀알엑스(30ml*3ea),709,6,272,0.383278,45.333333
1,9311062,(단종)멜라B_매트커버팩트_23베이지_본품_19g,1909,17,526,0.275441,30.941176
2,9310616,멜라_토닝원데이앰플_15ml,22504,100,1866,0.082916,18.660000
3,9309798,DWEGF_코어부스팅아이크림_30ml(홈쇼핑),9998,332,3437,0.343769,10.352410
4,9312895,[세트_임가공]멜라_토닝앰플쿠션_스페셜기획세트_21호(쿠팡기획세트),480,29,216,0.449688,7.448276


In [165]:
df_overview = df_overview[["자재", "저장 위치", "배치", "특별 재고", "기말 재고 수량"]].copy()
df_overview.rename(columns = {"자재":"자재코드", "저장 위치":"저장위치", "배치":"배치", "특별 재고":"특별재고", "기말 재고 수량":"기말재고"}, inplace=True)
df_overview["자재코드"] = df_overview["자재코드"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
df_overview["기말재고"] = pd.to_numeric(df_overview["기말재고"], errors="coerce").fillna(0)
#df_overview["기말재고"] = df_overview["기말재고"].astype(int)

In [166]:
display(df_overview)

,자재코드,저장위치,배치,특별재고,기말재고
0,1000940,NaN,B08393,O,79.863
1,1000940,NaN,B08471,O,607.564
2,1000940,NaN,B08677,O,404.540
3,1000940,NaN,B08682,O,147.002
4,1000940,NaN,B08683,O,166.405
...,...,...,...,...,...
3812,9401327,NaN,FC001,O,20.000
3813,9401327,5000.0,FC001,NaN,1962.000
3814,9401426,7000.0,ACB,NaN,1000.000
3815,9401548,5000.0,D260401001,NaN,300.000


In [167]:
df_overview.to_csv("checking1.csv", encoding = "utf-8-sig")

In [168]:
from inventory_utils2 import filter_special_stock

In [169]:
df_overview = filter_special_stock(df_overview)

In [170]:
df_overview.to_csv("checking2.csv", encoding = "utf-8-sig")

In [171]:
display(df_overview.head())

,자재코드,저장위치,배치,특별재고,기말재고
0,1000940,NaN,B08393,O,79.863
1,1000940,NaN,B08471,O,607.564
2,1000940,NaN,B08677,O,404.540
3,1000940,NaN,B08682,O,147.002
4,1000940,NaN,B08683,O,166.405


In [172]:
df_overview["자재코드"] = df_overview["자재코드"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
df_overview["저장위치"] = df_overview["저장위치"].fillna("알수없음")
df_overview["저장위치"] = df_overview["저장위치"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)

In [173]:
display(df_overview)

,자재코드,저장위치,배치,특별재고,기말재고
0,1000940,알수없음,B08393,O,79.863
1,1000940,알수없음,B08471,O,607.564
2,1000940,알수없음,B08677,O,404.540
3,1000940,알수없음,B08682,O,147.002
4,1000940,알수없음,B08683,O,166.405
...,...,...,...,...,...
1966,9401227,5000,D260401005,NaN,100.000
1967,9401231,5000,D260312002,NaN,2471.000
1968,9401327,5000,FC001,NaN,1962.000
1969,9401426,7000,ACB,NaN,1000.000


In [174]:
df_overview.to_csv("checking3.csv", encoding = "utf-8-sig")

In [175]:
df_overview = df_overview.groupby(["자재코드", "저장위치"], as_index = False).agg({
    "기말재고" : "sum"
})

In [176]:
# groupby 후 6080 관련 행 모두 출력
check_6080 = df_overview[df_overview["저장위치"].str.contains("6080", na=False)]
print(check_6080)
print()
# 자재코드+저장위치 중복 여부 확인
dups = df_overview[df_overview.duplicated(subset=["자재코드", "저장위치"], keep=False)]
print(f"중복 행 수: {len(dups)}")
print(dups[dups["저장위치"].str.contains("6080", na=False)])

         자재코드  저장위치   기말재고
28    2303248  6080  100.0
119   7301915  6080   50.0
145   7302264  6080  200.0
155   7302317  6080  523.0
184   9301703  6080    1.0
...       ...   ...    ...
1118  9312493  6080  331.0
1122  9312545  6080   83.0
1168  9313116  6080  119.0
1179  9313119  6080  129.0
1187  9313188  6080  280.0

[65 rows x 3 columns]

중복 행 수: 0
Empty DataFrame
Columns: [자재코드, 저장위치, 기말재고]
Index: []


In [177]:
df_overview.to_csv("checking4.csv", encoding = "utf-8-sig")

In [178]:
display(df_overview.head())

,자재코드,저장위치,기말재고
0,1000940,5000,117800.000
1,1000940,7000,0.000
2,1000940,알수없음,486084.942
3,1300271,알수없음,1201.960
4,2301915,5000,13.000


In [ ]:
df_overview = df_overview.pivot_table(
    index="자재코드",
    columns="저장위치",
    values="기말재고",
    aggfunc="sum",   # 중복 시 합계
    fill_value=0     # 없는 값은 0
)

In [180]:
display(df_overview.head())

저장위치,5000,5010,5100,5400,5600,6010,6020,6030,6040,6050,...,7000,7020,7030,7040,7050,7060,7070,7080,7090,알수없음
자재코드,,,,,,,,,,,,,,,,,,,,,
1000940,117800.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,486084.942
1300271,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1201.960
2301915,13.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000
2302226,2324.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000
2302396,13500.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000


In [181]:
df_overview.to_csv("check5.csv", encoding = "utf-8-sig")

In [182]:
df_standard = pd.merge(df_standard, df_overview, on = "자재코드", how = "left")

In [183]:
display(df_standard.head())

,자재코드,자재내역,3평판,SF,당월출하,판매율(평판),판매율(SF),5000,5010,5100,...,7000,7020,7030,7040,7050,7060,7070,7080,7090,알수없음
0,9302971,(단종)[임가공]RX_프레좀알엑스(30ml*3ea),709,6,272,0.383278,45.333333,657.0,0.0,0.0,...,0.0,0.0,4.0,53.0,0.0,0.0,0.0,0.0,0.0,0.0
1,9311062,(단종)멜라B_매트커버팩트_23베이지_본품_19g,1909,17,526,0.275441,30.941176,2178.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,69.0,0.0,12.0,0.0,0.0
2,9310616,멜라_토닝원데이앰플_15ml,22504,100,1866,0.082916,18.660000,2525.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,9309798,DWEGF_코어부스팅아이크림_30ml(홈쇼핑),9998,332,3437,0.343769,10.352410,35077.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,9312895,[세트_임가공]멜라_토닝앰플쿠션_스페셜기획세트_21호(쿠팡기획세트),480,29,216,0.449688,7.448276,1467.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [184]:
df_standard.to_csv("check6.csv", index = False, encoding = "utf-8-sig")